In [ ]:

from google.colab import files
import cv2
import matplotlib.pyplot as plt


# image upload

uploaded = files.upload()

# get image name

image_name = list(uploaded.keys())[0]

print("Uploaded Image:", image_name)

# read image

image = cv2.imread(image_name)

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# show image

plt.figure(figsize=(6,6))

plt.imshow(image)

plt.title("Uploaded Image")

plt.axis('off')

plt.show()



In [ ]:

import os

print(os.listdir('/content'))



In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

#use uploaded image

image = cv2.imread(image_name)

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#resize image

h, w = image.shape[:2]

while h > 1000 or w > 1000:
    h = h // 2
    w = w // 2

image = cv2.resize(image, (w, h))

#Green channel

green = image[:, :, 1]

#gaussian filter

blurred = cv2.GaussianBlur(
    green,
    (5, 5),
    0.9
)

# clahe

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8)
)

enhanced = clahe.apply(blurred)

#morphological opening

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5,5)
)

opened = cv2.morphologyEx(
    enhanced,
    cv2.MORPH_OPEN,
    kernel
)

# display results

plt.figure(figsize=(14,8))

plt.subplot(2,3,1)
plt.imshow(image)
plt.title("Original Image")
plt.axis('off')

plt.subplot(2,3,2)
plt.imshow(green, cmap='gray')
plt.title("Green Channel")
plt.axis('off')

plt.subplot(2,3,3)
plt.imshow(blurred, cmap='gray')
plt.title("Gaussian Filter")
plt.axis('off')

plt.subplot(2,3,4)
plt.imshow(enhanced, cmap='gray')
plt.title("CLAHE")
plt.axis('off')

plt.subplot(2,3,5)
plt.imshow(opened, cmap='gray')
plt.title("Morphological Opening")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:

# thresholding

_, thresh = cv2.threshold(
    opened,
    240,
    255,
    cv2.THRESH_BINARY
)

# canny edge detection

edges = cv2.Canny(
    opened,
    30,
    100
)

# Hough circle Transform

circles = cv2.HoughCircles(
    edges,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=50,
    param1=50,
    param2=20,
    minRadius=30,
    maxRadius=100
)

 # Draw detected circle

output = image.copy()

if circles is not None:

    circles = np.uint16(np.around(circles))

    for i in circles[0, :1]:

        center = (i[0], i[1])
        radius = i[2]

        # Outer Circle
        cv2.circle(
            output,
            center,
            radius,
            (0,255,0),
            3
        )

        # Center Point
        cv2.circle(
            output,
            center,
            2,
            (255,0,0),
            3
        )
# Display Results

plt.figure(figsize=(15,8))

plt.subplot(1,3,1)
plt.imshow(thresh, cmap='gray')
plt.title("Threshold")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(edges, cmap='gray')
plt.title("Canny Edge")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(output)
plt.title("Detected Optic Disc")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:

from skimage.segmentation import slic
from skimage.segmentation import mark_boundaries

# Slic superpixel segmentation

segments = slic(
    image,
    n_segments=100,
    compactness=10,
    start_label=1
)

# Draw superpixel Boundaries

superpixel_output = mark_boundaries(
    image,
    segments
)

# Display

plt.figure(figsize=(10,8))

plt.imshow(superpixel_output)

plt.title("SLIC Superpixel Segmentation")

plt.axis('off')

plt.show()


In [ ]:

# copy image

density_output = image.copy()

# Create circular roi mask

mask = np.zeros(green.shape, dtype=np.uint8)

height, width = green.shape

center_x = width // 2
center_y = height // 2

radius = 100

cv2.circle(
    mask,
    (center_x, center_y),
    radius,
    255,
    -1
)

# Apply mask

roi = cv2.bitwise_and(
    green,
    green,
    mask=mask
)

# canny edge detection

roi_edges = cv2.Canny(
    roi,
    30,
    100
)

# pixel density Count

pixel_count = cv2.countNonZero(roi_edges)

print("Pixel Density:", pixel_count)

# Draw roi circle

cv2.circle(
    density_output,
    (center_x, center_y),
    radius,
    (0,255,0),
    3
)

# display results

plt.figure(figsize=(15,8))

plt.subplot(1,3,1)
plt.imshow(mask, cmap='gray')
plt.title("ROI Mask")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(roi_edges, cmap='gray')
plt.title("ROI Edge Detection")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(density_output)
plt.title("Pixel Density Localization")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:

from skimage.segmentation import slic
from skimage.segmentation import mark_boundaries

# slic superpixel segmantation

segments = slic(
    image,
    n_segments=100,
    compactness=10,
    start_label=1
)

# create superpixel output

superpixel_output = mark_boundaries(
    image,
    segments
)

# show results

plt.figure(figsize=(8,8))

plt.imshow(superpixel_output)

plt.title("Superpixel Segmentation")

plt.axis('off')

plt.show()


In [ ]:

# final segmentation output

final_output = image.copy()

# Draw final detected disk

if circles is not None:

    circles = np.uint16(np.around(circles))

    for i in circles[0, :1]:

        center = (i[0], i[1])
        radius = i[2]

        # Final disk boundary
        cv2.circle(
            final_output,
            center,
            radius,
            (0,255,0),
            4
        )

        # center point
        cv2.circle(
            final_output,
            center,
            3,
            (255,0,0),
            4
        )
# Display final Results

plt.figure(figsize=(18,10))

plt.subplot(2,2,1)
plt.imshow(image)
plt.title("Original Fundus Image")
plt.axis('off')

plt.subplot(2,2,2)
plt.imshow(edges, cmap='gray')
plt.title("Canny Edge")
plt.axis('off')

plt.subplot(2,2,3)
plt.imshow(superpixel_output)
plt.title("Superpixel Segmentation")
plt.axis('off')

plt.subplot(2,2,4)
plt.imshow(final_output)
plt.title("Final Optic Disc Segmentation")
plt.axis('off')

plt.tight_layout()
plt.show()
